In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, ops
from keras.datasets import imdb
from keras.utils import pad_sequences

In [ ]:
np.random.seed(0)
tokens = ["The", "cat", "sat", "on", "the", "mat"]
d = 8
X = np.random.randn(len(tokens), d)                    # toy token embeddings
Wq, Wk, Wv = [np.random.randn(d, d) for _ in range(3)]
Q, K, V = X @ Wq, X @ Wk, X @ Wv

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

scores = Q @ K.T / np.sqrt(d)
weights = softmax(scores)                              # each row sums to 1
output = weights @ V
print("Attention weights shape:", weights.shape, "| row sums:", weights.sum(axis=1).round(2))

plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap="viridis")
plt.xticks(range(len(tokens)), tokens); plt.yticks(range(len(tokens)), tokens)
plt.colorbar(); plt.title("Self-attention weights"); plt.show()

In [ ]:
VOCAB, MAXLEN = 10000, 200
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB)
x_train = pad_sequences(x_train, maxlen=MAXLEN)
x_test = pad_sequences(x_test, maxlen=MAXLEN)

In [ ]:
class AttentionLayer(layers.Layer):
    """Additive (Bahdanau-style) attention over the time axis."""
    def build(self, input_shape):
        dim = input_shape[-1]
        self.W = self.add_weight(shape=(dim, dim), initializer="glorot_uniform", name="W")
        self.b = self.add_weight(shape=(dim,), initializer="zeros", name="b")
        self.u = self.add_weight(shape=(dim, 1), initializer="glorot_uniform", name="u")

    def attention_weights(self, h):                    # h: (batch, time, dim)
        score = ops.matmul(ops.tanh(ops.matmul(h, self.W) + self.b), self.u)   # (batch, time, 1)
        return ops.softmax(score, axis=1)

    def call(self, h):
        a = self.attention_weights(h)
        return ops.sum(h * a, axis=1)                  # context vector (batch, dim)

inp = layers.Input(shape=(MAXLEN,))
emb = layers.Embedding(VOCAB, 64)(inp)
h = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(emb)
att = AttentionLayer()
ctx = att(h)
out = layers.Dense(1, activation="sigmoid")(layers.Dropout(0.3)(ctx))

model = keras.Model(inp, out, name="BiLSTM_Attention")
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
hist = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.2, verbose=2)
print("Test accuracy:", round(model.evaluate(x_test, y_test, verbose=0)[1], 4))

plt.plot(hist.history["accuracy"], label="train"); plt.plot(hist.history["val_accuracy"], label="val")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.title("BiLSTM + Attention"); plt.show()

In [ ]:
encoder = keras.Model(inp, h)                         # sub-model that outputs hidden states
word_index = imdb.get_word_index()
idx2word = {v + 3: k for k, v in word_index.items()}
idx2word.update({0: "<pad>", 1: "<start>", 2: "<unk>"})

def show_attention(i, top=8):
    seq = x_test[i:i+1]
    states = encoder.predict(seq, verbose=0)
    w = ops.convert_to_numpy(att.attention_weights(states))[0, :, 0]
    words = [idx2word.get(t, "?") for t in seq[0]]
    p = float(model.predict(seq, verbose=0)[0][0])
    print(f"Review {i} | true={'pos' if y_test[i] else 'neg'} | predicted={'pos' if p > 0.5 else 'neg'} ({p:.2f})")
    best = [j for j in w.argsort()[::-1] if words[j] != "<pad>"][:top]
    print("Top attended words:", [(words[j], round(float(w[j]), 3)) for j in best])
    plt.figure(figsize=(12, 2.2)); plt.bar(range(MAXLEN), w, color="darkorange")
    plt.title("Attention weight per position"); plt.xlabel("token position"); plt.show()

for i in [0, 1]:
    show_attention(i)